# L4-2 按世界时查询台风（台风专线 T1）

<!-- course-meta: real-llm -->

这一步还原 huishang_demo 早期的做法：台风查询工具**只接受世界时**。
用户却总是用北京时提问。看看让模型自己减 8 小时会出什么问题。

工具调用的是真实的台风数据接口（getyphdata，仅内网可访问），接口本身只认北京时。
这一讲要用**真实模型**，`.env` 里的 `MAIN_API_KEY` 必须填好。
没有 API Key 时先看同目录的 `99_假模型剧本.ipynb`：那里用预设剧本演示同样的流程。

In [1]:
# 改了 common/、weather_tools/ 里的代码后自动重新加载，不用重启内核
%load_ext autoreload
%autoreload 2

import json

from langchain.agents import create_agent
from langchain_core.tools import tool

from common import llm
from common.trace import show, tool_calls
from weather_tools.agent_tools import get_beijing_time

## 1. 工具：只收世界时

下面是参考写法。注意 docstring 没有写清楚时区，模型调用是否会异常？

In [10]:
import requests

from weather_tools.timeutil import convert_tz
from weather_tools.typhoon import data_endpoint

# 真实接口地址由 data_endpoint() 提供（内网，不在讲义里明文给出）


@tool
def get_typhoon_data_utc(datetime: str) -> str:
    """查询西北太平洋活动台风，返回 JSON。

    参数 datetime：世界时，格式 YYYY-MM-DD HH:MM:SS。
    """
    # 此工具的参数要求输入参数是世界时（UTC），但在 docstring 里没有明确说明，模型可能会误以为是北京时（UTC+8），导致调用失败。
    datetime_bj = convert_tz(datetime, "%Y-%m-%d %H:%M:%S", "UTC", "UTC+8")
    url = data_endpoint(datetime_bj)
    try:
        resp = requests.get(url, timeout=20)
        resp.raise_for_status()
        return json.dumps(resp.json(), ensure_ascii=False)
    except requests.exceptions.Timeout:
        return f"查询超时：世界时 {datetime}"
    except requests.exceptions.ConnectionError:
        return "无法连接到台风数据服务（需在内网环境）"
    except requests.exceptions.HTTPError as exc:
        return f"HTTP 错误（{exc.response.status_code}）"
    except ValueError as exc:
        return f"返回数据非有效 JSON：{exc}"


print(get_typhoon_data_utc.invoke({"datetime": "2026-08-11 00:00:00"})[:200])
print(get_typhoon_data_utc.invoke({"datetime": "2026-08-11 08:00:00"})[:200])

{"status": 1, "tsid": [547, 550], "datetime": ["2026-08-11 08:00:00", "2026-08-11 08:00:00"], "data": [{"台风信息": "2026年第13号台风白海豚", "路径信息": [{"预报时间": "2026-08-11 08:00:00", "预报时效": "0小时", "经度": "115.6",
{"status": 1, "tsid": [547, 550], "datetime": ["2026-08-11 14:00:00", "2026-08-11 14:00:00"], "data": [{"台风信息": "2026年第13号台风白海豚", "路径信息": [{"预报时间": "2026-08-11 14:00:00", "预报时效": "0小时", "经度": "115.1",


同一天，传 00 时（世界时）查的是北京时 08 时；把北京时 08 时直接当世界时传进去，查的其实是北京时 16 时，两次返回的台风资料会不一样（时次不对，可能为空）。

## 2. 交给智能体

真实模型下请多问几次，统计换算对了几次。没有 API Key 时先看同目录的 `99_假模型剧本.ipynb`：那里用预设剧本（故意让模型“忘了换算”，演示常见错误）演示同样的流程。

In [8]:
model = llm.main_llm()
agent = create_agent(model=model, tools=[get_beijing_time, get_typhoon_data_utc],
                     system_prompt="你是一个气象助手。")
r = agent.invoke({"messages": [{"role": "user", "content": "今天上午有没有发布台风预报？"}]})
show(r)

[1] 用户：今天上午有没有发布台风预报？

[2] 模型决定调用工具 get_beijing_time，参数 {}

[3] 工具 get_beijing_time 返回：2026-10-29 10:20

[4] 模型决定调用工具 get_typhoon_data_utc，参数 {'datetime': '2026-10-29 00:00:00'}
[4] 模型决定调用工具 get_typhoon_data_utc，参数 {'datetime': '2026-10-28 18:00:00'}

[5] 工具 get_typhoon_data_utc 返回：{"status": 1, "tsid": [], "datetime": [], "data": []}

[6] 工具 get_typhoon_data_utc 返回：{"status": 1, "tsid": [], "datetime": [], "data": []}

[7] 模型回答：**截至北京时间 2026-10-29 10:20，没有台风预报可发布。**

查询结果如下：

| 查询时刻（世界时 UTC） | 对应北京时间 | 西北太平洋活动台风 |
|---|---|---|
| 2026-10-28 18:00 | 10-29 02:00（今晨） | 无（台风编号列表为空） |
| 2026-10-29 00:00 | 10-29 08:00（今早） | 无（台风编号列表为空） |

也就是说，今天凌晨到今天上午，西北太平洋没有处于活动状态的台风，因此气象部门无需发布台风预报或台风预警。

补充说明：这里查询的是西北太平洋活动台风的编号和报文数据；如果该海区有台风活动，通常会出现对应的台风编号和预报内容，而当前返回的编号列表为空。如果你想确认是否有“热带低压”或“台风胚胎”等未编号系统的消息，或想查看其他海域的情况，可以告诉我。

另外，发布时间可能还有少量滞后，因此建议稍后（例如 11 时前后）再核对一次最新报文。



## 4. 练习

1. 如何修改系统提示词、用户提示词，或者工具说明，让模型记住“用户说的时间都是北京时”，并在调用工具前自动减 8 小时？
2. 改造工具，让它能接受北京时和世界时两种输入，返回的结果里标明是哪个时次。问“今天早上 8 点有没有台风？”，看看模型会不会用这个新工具。